# 08 — Discount Curves

All pricing in `derivatives_pricing` is driven by a `DiscountCurve` that
provides the risk-free discount factor $D(t)$.

This notebook demonstrates the three factory methods for building curves
and shows how curve shape affects option prices.

## 1) Setup

In [1]:
import datetime as dt

import numpy as np
import derivatives_pricing as dp

In [2]:
pricing_date = dt.datetime(2025, 6, 15)
maturity = dt.datetime(2026, 6, 15)

## 2) Flat Curve

`DiscountCurve.flat(rate, end_time)` builds a curve from a single
continuously-compounded rate: $D(t) = e^{-rt}$.

In [3]:
flat_curve = dp.DiscountCurve.flat(rate=0.05, end_time=5.0)

# Discount factors at various maturities
for t in [0.25, 0.5, 1.0, 2.0, 5.0]:
    print(f"  D({t:.2f}) = {float(flat_curve.df(t)):.6f}")

  D(0.25) = 0.987578
  D(0.50) = 0.975310
  D(1.00) = 0.951229
  D(2.00) = 0.904837
  D(5.00) = 0.778801


## 3) Curve from Forward Rates

`DiscountCurve.from_forwards(times, forwards)` builds a curve from
piecewise-constant forward rates.

In [4]:
times = np.array([0, 0.5, 1.0, 2.0, 5.0])
forwards = np.array([0.04, 0.045, 0.05, 0.055])  # upward-sloping forwards

fwd_curve = dp.DiscountCurve.from_forwards(times=times, forwards=forwards)

print("Forward-rate curve:")
for t in [0.25, 0.5, 1.0, 2.0, 5.0]:
    print(f"  D({t:.2f}) = {float(fwd_curve.df(t)):.6f}")

Forward-rate curve:
  D(0.25) = 0.990050
  D(0.50) = 0.980199
  D(1.00) = 0.958390
  D(2.00) = 0.911649
  D(5.00) = 0.772982


## 4) Curve from Zero Rates

`DiscountCurve.from_zero_rates(times, zero_rates)` builds a curve from
continuously-compounded spot (zero) rates.

In [5]:
times_z = np.array([0, 0.5, 1.0, 2.0, 5.0])
zero_rates = np.array([0.035, 0.04, 0.045, 0.05, 0.055])

zero_curve = dp.DiscountCurve.from_zero_rates(times=times_z, zero_rates=zero_rates)

print("Zero-rate curve:")
for t in [0, 0.25, 0.5, 1.0, 2.0, 5.0]:
    print(f"  D({t:.2f}) = {float(zero_curve.df(t)):.6f}")

Zero-rate curve:
  D(0.00) = 1.000000
  D(0.25) = 0.990050
  D(0.50) = 0.980199
  D(1.00) = 0.955997
  D(2.00) = 0.904837
  D(5.00) = 0.759572


## 5) Effect on Option Prices

Higher rates increase call values and decrease put values (via the
risk-neutral drift and discounting).

In [6]:
spec_call = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

spec_put = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

print(f"{'Curve':<20} {'Call':>10} {'Put':>10}")
print("-" * 42)
for label, crv in [
    ("Flat 3%", dp.DiscountCurve.flat(rate=0.03, end_time=2.0)),
    ("Flat 5%", dp.DiscountCurve.flat(rate=0.05, end_time=2.0)),
    ("Flat 8%", dp.DiscountCurve.flat(rate=0.08, end_time=2.0)),
    ("Forward (upward)", fwd_curve),
    ("Zero rate", zero_curve),
]:
    mkt = dp.MarketData(pricing_date=pricing_date, discount_curve=crv, currency="USD")
    und = dp.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=mkt)

    call_pv = dp.OptionValuation(
        underlying=und, spec=spec_call, pricing_method=dp.PricingMethod.BSM
    ).present_value()
    put_pv = dp.OptionValuation(
        underlying=und, spec=spec_put, pricing_method=dp.PricingMethod.BSM
    ).present_value()
    print(f"{label:<20} {call_pv:>10.4f} {put_pv:>10.4f}")

Curve                      Call        Put
------------------------------------------
Flat 3%                  9.4134     6.4580
Flat 5%                 10.4506     5.5735
Flat 8%                 12.1058     4.4175
Forward (upward)        10.0552     5.8942
Zero rate               10.1861     5.7859


## 6) Non-Flat Curves
Non-flat curves are fully supported across all deterministic pricing methods.

In [11]:
market_fwd = dp.MarketData(pricing_date=pricing_date, discount_curve=fwd_curve, currency="USD")
underlying_fwd = dp.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=market_fwd)

bsm_pv = dp.OptionValuation(
    underlying=underlying_fwd, spec=spec_call, pricing_method=dp.PricingMethod.BSM
).present_value()

binom_pv = dp.OptionValuation(
    underlying=underlying_fwd,
    spec=spec_call,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=500),
).present_value()

pde_pv = dp.OptionValuation(
    underlying=underlying_fwd,
    spec=spec_call,
    pricing_method=dp.PricingMethod.PDE_FD,
    params=dp.PDEParams(),
).present_value()

print("Forward curve — European call:")
print(f"  BSM:      {bsm_pv:.4f}")
print(f"  Binomial: {binom_pv:.4f}")
print(f"  PDE:      {pde_pv:.4f}")

Forward curve — European call:
  BSM:      10.0552
  Binomial: 10.0512
  PDE:      10.0453


## 7) Monte Carlo with Non-Flat Curves

`GBMProcess` also supports non-flat discount curves. Internally, the path
generator computes forward rates and dividend yields from the provided
curves at each time step — so the drift varies over the life of the option.

In [12]:
maturity_mc = dt.datetime(2026, 6, 15)
sim_config = dp.SimulationConfig(paths=200_000, end_date=maturity_mc, num_steps=200)

gbm_fwd = dp.GBMProcess(
    market_data=market_fwd,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.20),
    sim_config=sim_config,
)

mc_pv = dp.OptionValuation(
    underlying=gbm_fwd,
    spec=spec_call,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
).present_value()

print("Forward curve — European call (MC vs BSM):")
print(f"  BSM: {bsm_pv:.4f}")
print(f"  MC:  {mc_pv:.4f}")

Forward curve — European call (MC vs BSM):
  BSM: 10.0552
  MC:  10.0517
